# [0] Naive Trial with MaxPooling with EXAONE

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device Number
DEVICE_NUM = 7
ADDITIONAL_GPU = 0

from os import environ
environ["CUDA_VISIBLE_DEVICES"] = ",".join([f"{i+DEVICE_NUM}" for i in range(0, ADDITIONAL_GPU+1)])
environ["CUDA_VISIBLE_DEVICES"]

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset, BalancedDataLoader

from transformers import PreTrainedModel, AutoModelForCausalLM, AutoConfig, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch import nn, optim
import torch

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc
import re

In [ ]:
from sklearn.exceptions import UndefinedMetricWarning
import warnings

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

In [ ]:
# Set CUDA Device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}")

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
# project name
PROJECT_NAME = "0_naive_maxpool"

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1, balancing_ratio=1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1, balancing_ratio=1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

In [ ]:
# check label distribution
len([i for i in train_dataset if i[1] == 0]), len([i for i in train_dataset if i[1] == 1])

#### Paragraph splitting test

In [ ]:
def document_to_paragraphs(doc):
    result = doc.split("\n\n")
    if len(result) > 1:
        return [res.strip() for res in result]
    result = doc.split("\n")
    if len(result) > 1:
        return [res.strip() for res in result]
    if len(doc) > 650:
        sentences = re.split(r'(?<=[.?!])\s*', doc)
        sentences = [s.strip() for s in sentences if s]
        return [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]  # group by 3 sentences
    else:
        return [doc.strip()]

In [ ]:
converted = [(document_to_paragraphs(document[0]), document[1]) for document in train_dataset]
conv_size = []
for document, label in converted:
    conv_size.append(len(document))
    if len(document) == 1:
        print(f"ERROR: Document with single paragraph - {document[0]}", file=sys.stderr, flush=True)
    elif len(document) > 50:
        print(f"WARNING: Document with too many paragraphs - {len(document)} paragraphs", flush=True)
    else:
        pass

## Define Model

In [ ]:
base_model_id = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_quant_type="fp8"
)

In [ ]:
from typing import Optional

class ExaoneForNaiveTextDetection(PreTrainedModel):
    def __init__(
        self,
        quantization_config: Optional[BitsAndBytesConfig] = None
    ):
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            quantization_config=quantization_config
        )
        super().__init__(base.config)

        # 0. Register base model
        self.base = base.transformer
        for param in self.base.parameters():
            param.requires_grad = False  # Freeze the base model parameters

        # 1. Add MaxPooling layer
        self.pool = nn.AdaptiveMaxPool1d(1)

        # 2. Final classification layer
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.config.hidden_size, 1),
        ).to(dtype=torch.bfloat16)

        # 3. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,  # WARNING: This forward function does not support 'real' batch calculation (paragraphs are batched)
        input_ids: torch.LongTensor,  # a paragraph of sentences
        attention_mask: torch.Tensor  # attention mask for each paragraph
    ) -> torch.Tensor:
        with torch.no_grad():
            hidden_states = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        pooled = self.pool(hidden_states.transpose(1, 2)).squeeze(-1)
        return self.classifier(pooled)

In [ ]:
try:  # For the case of reloading the model class
    model.__class__ = ExaoneForNaiveTextDetection
except Exception:
    pass

In [ ]:
model = ExaoneForNaiveTextDetection(quantization_config=None)
try:
    from safetensors.torch import load_file
    state_dict = load_file(f"./models/{PROJECT_NAME}_last/model.safetensors")
    model.load_state_dict(state_dict)
except Exception:
    pass
model = model.bfloat16()
model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

### Utils

In [ ]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def calc_score(lb, pd):
    pd_discrete = to_label(pd)
    acc = accuracy_score(lb, pd_discrete)
    f1 = f1_score(lb, pd_discrete)
    rocauc = roc_auc_score(lb, pd)
    return f"ACC: {acc:.6f}, F1: {f1:.6f}, ROCAUC: {rocauc:.6f}"

In [ ]:
BATCH_SIZE = 1, 1, 1
GRADIENT_ACCUMULATION_STEPS = 8  # real batch

train_loader = BalancedDataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: x)

In [ ]:
EPOCHS = 20
START_EPOCH = 0
LEARNING_RATE = 2e-5, 1e-6

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE[0])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, min_lr=LEARNING_RATE[1])

### Training Loop

In [ ]:
with (
    tqdm(range(START_EPOCH, START_EPOCH+EPOCHS), desc="[Running Epochs]") as epochs,
    tqdm(range(len(train_dataset)//BATCH_SIZE[0]), desc="[Training]") as train_progress,
    tqdm(range(len(valid_dataset)//BATCH_SIZE[1]), desc="[Validating]") as valid_progress
):
    for epoch in epochs:
        train_progress.reset()
        train_loss, train_preds, train_labels = [], [], []

        # Train
        model.train()
        for step, (texts, labels) in enumerate(train_loader):
            texts, labels = texts[0], labels  # Unpack single batch
            try:
                logits = model(**tokenize(document_to_paragraphs(texts)))[0]
                loss = criterion(logits, labels.float().to(device)) #* (1.4 if labels[0].item() == 1 else 1)
                train_preds.append(torch.sigmoid(logits)[0].item())
                train_labels.append(labels.item())
                train_loss.append(loss.item())
                loss.backward()

                if (step+1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()

                train_progress.update(1)
                train_progress.set_description(f"[Training] Step: {step+1}, Loss: {sum(train_loss)/len(train_loss):.6f}, " + calc_score(train_labels, train_preds))
            except Exception as e:
                print(e, file=sys.stderr)

        # Validate
        model.eval()
        valid_loss, valid_preds, valid_labels = [], [], []
        torch.cuda.empty_cache(); gc.collect(); valid_progress.reset()
        for texts, labels in valid_loader:
            texts, labels = texts[0], labels  # Unpack single batch
            try:
                with torch.no_grad():
                    logits = model(**tokenize(document_to_paragraphs(texts)))[0]
                    loss = criterion(logits, labels.float().to(device)) #* (1.4 if labels[0].item() == 1 else 1)
                    scores = torch.sigmoid(logits)[0]
                    preds = to_label(scores)

                    valid_preds.append(scores[0].item())
                    valid_labels.append(labels.item())
                    valid_loss.append(loss)
            except Exception as e:
                print(e, file=sys.stderr)

            valid_progress.update(1)
            valid_progress.set_description(f"[Validating] Loss: {torch.mean(torch.stack(valid_loss)):.6f}, " + calc_score(valid_labels, valid_preds))

        model.save_pretrained(f"./models/{PROJECT_NAME}_{epoch}")
        model.save_pretrained(f"./models/{PROJECT_NAME}_last")
        scheduler.step(torch.mean(torch.stack(valid_loss)))

In [ ]:
labels

In [ ]:
# Model Saving
model.save_pretrained(f"./models/{PROJECT_NAME}_last")

### Final Output

In [ ]:
per_titles = {}
for i, row in test_dataset.raw.iterrows():
    if row['title'] not in per_titles:
        per_titles[row['title']] = [row['paragraph_text']]
    else:
        per_titles[row['title']].append(row['paragraph_text'])
test_dataset_bundled = list(per_titles.values())
test_dataset_bundled

In [ ]:
results = []
with tqdm(test_dataset_bundled, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for texts in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in texts]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)
                results.extend(scores.tolist())
                for preds in to_label(scores):
                    if preds == 0:
                        humans += 1
                    else:
                        ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

len(results) == len(test_dataset)

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x="generated", kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv(f"./data/submission_{PROJECT_NAME[2:]}.csv", index=False, encoding='utf-8-sig')